<a href="https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

Research question: which content pages should an editor with limited review capacity look at first? (Lane 2: Refresh / Content Opportunity Scoring.) Decision supported: weekly prioritization for a content/SEO editor. Action: refresh, rewrite, or fix metadata on flagged pages. Cost of a wrong call: reviewing a healthy page wastes scarce editor time; missing a genuinely declining high-traffic page lets visibility erode further unnoticed.

In [7]:
!git clone https://github.com/SUYAIBALSIFAT/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 175 (delta 76), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 1.89 MiB | 5.41 MiB/s, done.
Resolving deltas: 100% (76/76), done.
/content/flyrank-ml-internship/flyrank-ml-internship


## 2. Data

Source: FlyRank ML Internship starter dataset, 30,000 anonymized content pages, 90-day search performance + engagement + freshness fields. Excluded from features: trend_direction, trend_pct (label-derived), content_id/client_id (grouping only, never features). No client names, URLs, or raw queries anywhere.

In [8]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["needs_review"] = ((df.trend_direction == "down") & (df.impressions_90d >= 100)).astype(int)

print("Total pages:", len(df))
print("Positive rate (needs_review proxy):", round(df.needs_review.mean(), 4))

Total pages: 30000
Positive rate (needs_review proxy): 0.4384


## 3. Methodology

Label/proxy: needs_review = 1 if trend_direction == "down" AND impressions_90d >= 100. A same-window proxy, not a validated future outcome.
Baseline (Week 4): transparent rule — flag declining pages with demand, or visible pages under-capturing clicks (CTR < 0.5, position 1-20, impressions_90d >= 500). Two signals checked first: staleness (MIXED verdict, dropped — small n=174, opposite direction), low CTR (CONFIRMED, kept — n=9,759 vs 2,264, real gap).
Model (Week 5): Logistic Regression + Random Forest, 20 numeric features, client-grouped split (GroupShuffleSplit, 75/25) — confirmed zero client overlap between train/test.
Leakage audit (Week 6): trained with/without impressions_90d — score identical (0.66 both ways), confirming no leakage. No product-decision flags exist in this dataset to accidentally include.

In [9]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

feature_cols = ["search_volume","competition","cpc","word_count","char_count",
    "impressions_90d","clicks_90d","sessions_90d","users_90d","engaged_sessions_90d",
    "ai_sessions_90d","scroll_events_90d","days_with_impressions","days_with_sessions",
    "content_age_days","days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct"]

data = df.dropna(subset=feature_cols + ["client_id"]).copy()
X = data[feature_cols].fillna(0)
y = data["needs_review"]
groups = data["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))
X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

print("client overlap (must be 0):", len(set(groups.iloc[tr_idx]) & set(groups.iloc[te_idx])))

client overlap (must be 0): 0


## 4. Results (vs baseline)

Reported on the same grouped test split as the baseline. Random-split number shown separately as the leaked/misleading control, not a real result.

In [10]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_declining = (data.trend_direction == "down") & (data.impressions_90d >= 100)
base_low_ctr = (data.impressions_90d >= 500) & (data.avg_position.between(0, 20, inclusive="right")) & (data.ctr < 0.5)
baseline_score = (data["impressions_90d"] * (base_declining.astype(int) + base_low_ctr.astype(int))).iloc[te_idx].values
p50_base = precision_at_k(baseline_score, y_te.values, 50)

scaler = StandardScaler().fit(X_tr)
log_reg = LogisticRegression(max_iter=2000).fit(scaler.transform(X_tr), y_tr)
lr_probs = log_reg.predict_proba(scaler.transform(X_te))[:, 1]
p50_lr = precision_at_k(lr_probs, y_te.values, 50)
auc_lr = roc_auc_score(y_te, lr_probs)

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
rf_probs = rf.predict_proba(X_te)[:, 1]
p50_rf = precision_at_k(rf_probs, y_te.values, 50)
auc_rf = roc_auc_score(y_te, rf_probs)

from sklearn.model_selection import train_test_split
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf_leak = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr_r, ytr_r)
p50_leak = precision_at_k(rf_leak.predict_proba(Xte_r)[:, 1], yte_r.values, 50)

results = pd.DataFrame({
    "method": ["baseline rule", "logistic regression", "random forest (grouped/honest)", "random forest (random split/leaked)"],
    "precision@50": [p50_base, p50_lr, p50_rf, p50_leak],
})
results

,method,precision@50
0,baseline rule,0.74
1,logistic regression,0.90
2,random forest (grouped/honest),0.66
3,random forest (random split/leaked),1.00


## 5. Limitations

Proxy label, not a validated future outcome. Baseline shares logic with the label, weakening the comparison's strength. Single dataset snapshot, limited clients, one point in time. No causal claims — observational only. 19 pages in the Week-7 queue showed model/rule disagreement (model scored ≥0.90, rule said "monitor") — a reviewer should know this cutoff exists.

## 6. Ranked recommendations

1. Refresh "declining_and_low_ctr" pages first — the double-signal case.
2. Treat CTR = 0.00 rows as a data-quality check first, not an automatic edit.
3. Route "ctr_review_candidate" pages to snippet/metadata fixes, not full rewrites.
4. Re-check borderline CTR calls (near the 0.5 cutoff) before committing review time.
Never automate: publishing, deletion, or rewriting from this score alone.

## 7. Artifacts the paper embeds

Regenerating the queue, metrics JSON, and both figures used in the deployed paper.

In [11]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.metrics import roc_curve, confusion_matrix

matplotlib.rcParams['font.family'] = 'DejaVu Sans'

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# --- Rebuild the queue + reason codes on this notebook's rf/data_te ---
data_te = data.iloc[te_idx].copy()
data_te["model_score"] = rf.predict_proba(X_te)[:, 1]

def reason_code(row):
    declining = (row.trend_direction == "down") and (row.impressions_90d >= 100)
    low_ctr = (row.impressions_90d >= 500) and (0 < row.avg_position <= 20) and (row.ctr < 0.5)
    if declining and low_ctr: return "declining_and_low_ctr"
    elif declining: return "declining_with_demand"
    elif low_ctr: return "ctr_review_candidate"
    return "monitor"

data_te["reason_code"] = data_te.apply(reason_code, axis=1)
data_te["action"] = data_te["reason_code"].map({
    "declining_and_low_ctr": "refresh_review", "declining_with_demand": "refresh_review",
    "ctr_review_candidate": "ctr_review", "monitor": "monitor"})

queue = data_te[data_te.action != "monitor"].sort_values("model_score", ascending=False)
out_cols = ["content_id", "model_score", "reason_code", "action", "impressions_90d", "avg_position", "ctr"]
queue[out_cols].to_csv("work/outputs/content_action_playbook.csv", index=False)

# --- ROC curve data ---
fpr_lr, tpr_lr, _ = roc_curve(y_te, lr_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_te, rf_probs)

# --- Confusion matrix at top-50 threshold (RF) ---
order = np.argsort(-rf_probs)
top50_idx = order[:50]
y_pred_top50 = np.zeros(len(y_te), dtype=int)
y_pred_top50[top50_idx] = 1
cm = confusion_matrix(y_te, y_pred_top50)
tn, fp, fn, tp = cm.ravel()

# --- Metrics receipts ---
metrics = {
    "queue_size": int(len(queue)),
    "test_set_size": int(len(data_te)),
    "baseline_precision_at_50": round(float(p50_base), 3),
    "logreg_precision_at_50": round(float(p50_lr), 3),
    "logreg_roc_auc": round(float(auc_lr), 3),
    "rf_precision_at_50_grouped_split": round(float(p50_rf), 3),
    "rf_roc_auc": round(float(auc_rf), 3),
    "rf_precision_at_50_random_split_leaked": round(float(p50_leak), 3),
    "base_rate": round(float(y_te.mean()), 3),
    "confusion_matrix_top50": {"TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn)},
    "high_score_rule_disagreement_count": int(((data_te.model_score >= 0.9) & (data_te.action == "monitor")).sum()),
}
with open("work/outputs/w08_capstone_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Exported queue + metrics:", metrics)

# --- Chart 1: precision comparison ---
fig, ax = plt.subplots(figsize=(7, 4.2))
methods = ['Baseline\nrule', 'Logistic\nRegression', 'Random Forest\n(grouped split)', 'Random Forest\n(random split —\nmisleading)']
values = [metrics["baseline_precision_at_50"], metrics["logreg_precision_at_50"],
          metrics["rf_precision_at_50_grouped_split"], metrics["rf_precision_at_50_random_split_leaked"]]
colors = ['#8a8a8a', '#2f5d50', '#2f5d50', '#c98a3e']
bars = ax.bar(methods, values, color=colors, width=0.55)
ax.axhline(metrics["base_rate"], color='#c0392b', linestyle='--', linewidth=1.2, label=f'Base rate ({metrics["base_rate"]})')
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Precision@50')
ax.set_ylim(0, 1.12)
ax.set_title('Precision@50: baseline vs. model, honest vs. leaked split', fontsize=11)
ax.legend(loc='upper left', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig("work/figures/precision_comparison.png", dpi=160)
plt.close()

# --- Chart 2: feature importance ---
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False).head(6)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(importances.index[::-1], importances.values[::-1], color='#2f5d50', height=0.55)
ax.set_xlabel('Random Forest feature importance')
ax.set_title('What the model leans on', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig("work/figures/feature_importance.png", dpi=160)
plt.close()

# --- Chart 3: ROC curve ---
fig, ax = plt.subplots(figsize=(6, 5.2))
ax.plot(fpr_lr, tpr_lr, color='#2f5d50', linewidth=2, label=f"Logistic Regression (AUC={auc_lr:.3f})")
ax.plot(fpr_rf, tpr_rf, color='#c98a3e', linewidth=2, label=f"Random Forest (AUC={auc_rf:.3f})")
ax.plot([0, 1], [0, 1], color='#999', linestyle='--', linewidth=1, label='Chance (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC curve — client-grouped test split', fontsize=11)
ax.legend(loc='lower right', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig("work/figures/roc_curve.png", dpi=160)
plt.close()

# --- Chart 4: confusion matrix at top-50 ---
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.imshow(cm, cmap='Greens', alpha=0.85)
labels = [[f'TN\n{cm[0,0]}', f'FP\n{cm[0,1]}'], [f'FN\n{cm[1,0]}', f'TP\n{cm[1,1]}']]
for i in range(2):
    for j in range(2):
        ax.text(j, i, labels[i][j], ha='center', va='center', fontsize=13, fontweight='bold',
                color='white' if cm[i, j] > cm.max() * 0.5 else '#1a1a1a')
ax.set_xticks([0, 1]); ax.set_xticklabels(['Predicted: not top-50', 'Predicted: top-50'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['Actual: no review needed', 'Actual: needs review'])
ax.set_title('Confusion matrix — Random Forest, top-50 review capacity', fontsize=10.5)
plt.tight_layout()
plt.savefig("work/figures/confusion_matrix.png", dpi=160)
plt.close()

print("All 4 charts saved to work/figures/")

Exported queue + metrics: {'queue_size': 3541, 'test_set_size': 5745, 'baseline_precision_at_50': 0.74, 'logreg_precision_at_50': 0.9, 'logreg_roc_auc': 0.815, 'rf_precision_at_50_grouped_split': 0.66, 'rf_roc_auc': 0.815, 'rf_precision_at_50_random_split_leaked': 1.0, 'base_rate': 0.491, 'confusion_matrix_top50': {'TP': 33, 'FP': 17, 'FN': 2785, 'TN': 2910}, 'high_score_rule_disagreement_count': 19}
All 4 charts saved to work/figures/


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.


Demo outline

5-minute demo: (1) show the baseline rule and its 0.74 precision@50 — "here's the honest floor." (2) Show the model result — 0.66 grouped vs 1.0 random split — "here's why validation design matters more than the algorithm." (3) Show the top of the ranked queue and one CTR=0.00 outlier — "here's what a reviewer actually sees." (4) End on limitations — proxy label, no causal claim.

Social post cut



I built a content-refresh priority model on 30K real search pages. A simple rule hit 74% precision@50 — a Random Forest looked "perfect" (100%!) under a naive test split, but that was memorization. Under an honest client-grouped split, it dropped to 66%. The real finding wasn't "model beats rule" — it was how easily validation lies to you.

Employer 3-sentencer

I built a content-prioritization model on a 30K-page real search dataset, comparing a transparent baseline rule against Logistic Regression and Random Forest under a client-grouped validation split. The honest result showed a naive random split inflating Random Forest's score from 0.66 to a misleading 1.0 — a finding about validation design, not just model performance. The output is a decision-support ranked queue for content editors, with full leakage auditing and honest-claim framing throughout.